# 04d - QC 済み細胞・遺伝子の original-scale merge ＋ integrated QC

このノートブックは、04a〜04c で作成した `*.stage1_filtered.h5ad` を読み込み、
**前処理を揃えずに** outer / inner merge を作る。その後 merge 後の QC 確認指標を計算・可視化し、
プロットを見て手動で追加 QC 条件を決め、final QC 後の outer / inner を保存する。

## 実行順序

```
1. 04a〜04cの *.stage1_filtered.h5ad を読み込む
2. outer / inner でmergeする
3. ただし、ここではまだh5ad保存しない
4. merge後QC確認指標を計算する
5. QC指標をCSV保存する
6. QC指標を可視化する
7. プロットを見て、追加QC条件を手動で決める
8. 追加QCを実行し、qc_pass_integrated / qc_reason_integrated / qc_pass_final を作る
9. qc_pass_final == True の細胞だけに絞った outer / inner を保存する
```

## このノートブックで保存する merge の意味（重要）

```
この merged_qc_original_scale は、前処理を揃えた解析用mergeではない。

.X は raw_count_like、cpm_tpm_like、log_normalized_like が混在している。

matrix_sum は混在スケールでは意味がないため計算しない。

n_nonzero_genes と n_nonzero_mt_genes は、現在の .X における検出遺伝子数として解釈する。

pct_mt_for_qc は qc_preprocessing_state に応じて計算する。
raw_count_like と cpm_tpm_like では現在の .X からそのまま割合を計算する。
log_normalized_like では np.expm1 で戻してから割合を計算する。

ただし、log_normalized_like の pct_mt_for_qc は、全遺伝子が残っていて、かつ log1p(normalized_counts) 以外の変換が入っていない場合にのみ妥当である。

scale、regress_out、HVG subset後の行列、負の値を含む行列ではこのpct_mt復元は使わない。
```

混在スケールのため、merged h5ad 全体に対して `total_counts` や `matrix_sum` を同じ意味で解釈してはいけない。
raw count だけを使いたい場合は `qc_preprocessing_state == "raw_count_like"` で subset する。

このノートブックでは `normalize_total` / `log1p` / `scale` / `regress_out` は実行しない。


## セットアップ（import・パス）

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("../../").resolve()

QC_H5AD_DIR = ROOT / "data" / "qc_h5ad"
MERGED_DIR = ROOT / "data" / "merged_h5ad"

RESULTS_ROOT = ROOT / "results" / "qc_original_scale_pipeline"
OUT_DIR = RESULTS_ROOT / "04d_merged_original_scale"
PLOT_DIR = OUT_DIR / "plots"

MERGED_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

STATES = ["raw_count_like", "cpm_tpm_like", "log_normalized_like"]
OBS_KEEP = [
    "cell_uid", "source_accession", "dataset_id", "Condition",
    "qc_preprocessing_state", "qc_pass_stage1", "qc_reason_stage1",
]
MAX_PLOT = 100_000          # 可視化用サンプリング上限
RNG = np.random.default_rng(0)

print("ROOT       :", ROOT)
print("qc h5ad dir:", QC_H5AD_DIR)
print("merged dir :", MERGED_DIR)
print("out dir    :", OUT_DIR)


ROOT       : /home/suzuki/Learn/SMA/v2
qc h5ad dir: /home/suzuki/Learn/SMA/v2/data/qc_h5ad
merged dir : /home/suzuki/Learn/SMA/v2/data/merged_h5ad
out dir    : /home/suzuki/Learn/SMA/v2/results/qc_original_scale_pipeline/04d_merged_original_scale


## 1. 04a〜04c の `*.stage1_filtered.h5ad` を読み込む

obs は最小限（`OBS_KEEP`）に絞り、var_names を `gene_symbol_upper` に揃える。

In [2]:
def get_gene_symbols_upper(adata):
    if "gene_symbol_upper" in adata.var.columns:
        return adata.var["gene_symbol_upper"].astype(str).str.upper()
    if "gene_symbol" in adata.var.columns:
        return adata.var["gene_symbol"].astype(str).str.upper()
    return pd.Index(adata.var_names).astype(str).str.upper()


def on_symbol_upper(a):
    """var_names を gene_symbol_upper に揃える。"""
    b = a.copy()
    b.var_names = pd.Index(get_gene_symbols_upper(b))
    b.var_names_make_unique()
    return b


adatas = []
for state in STATES:
    sdir = QC_H5AD_DIR / state
    files = sorted(sdir.glob("*.stage1_filtered.h5ad"))
    print(f"{state}: {len(files)} filtered h5ad")
    for p in files:
        a = sc.read_h5ad(p)
        if "qc_preprocessing_state" not in a.obs.columns:
            a.obs["qc_preprocessing_state"] = state
        if "cell_uid" not in a.obs.columns:
            a.obs["cell_uid"] = a.obs_names.astype(str)
        keep = [c for c in OBS_KEEP if c in a.obs.columns]
        a.obs = a.obs[keep].copy()
        adatas.append(on_symbol_upper(a))

print("\ntotal adatas to merge:", len(adatas))


raw_count_like: 9 filtered h5ad
cpm_tpm_like: 1 filtered h5ad
log_normalized_like: 1 filtered h5ad

total adatas to merge: 11


## 2. outer / inner で merge（まだ保存しない）

`obs_names` はすでに `cell_uid` の前提なので `index_unique=None`（二重 prefix しない）。
ここでは `normalize_total` / `log1p` / `scale` は実行しない → `.X` は混在スケール。

In [3]:
merged_outer = ad.concat(
    adatas,
    join="outer",
    merge="same",
    index_unique=None,
)

merged_inner = ad.concat(
    adatas,
    join="inner",
    merge="same",
    index_unique=None,
)

if not merged_outer.obs_names.is_unique:
    print("WARNING: merged_outer obs_names (cell_uid) is not unique!")
if not merged_inner.obs_names.is_unique:
    print("WARNING: merged_inner obs_names (cell_uid) is not unique!")

print("merged_outer:", merged_outer.shape)
print("merged_inner:", merged_inner.shape)
print(merged_outer.obs["qc_preprocessing_state"].value_counts())


merged_outer: (277375, 34911)
merged_inner: (277375, 8863)
qc_preprocessing_state
raw_count_like         255474
log_normalized_like     21575
cpm_tpm_like              326
Name: count, dtype: int64


## 3. QC 確認指標の関数

- `get_mt_mask`: gene name を大文字化し `MT-` で始まるかで MT 遺伝子を判定（human `MT-` / mouse `Mt-` 両対応）。
- `add_nonzero_metrics`: 現在の `.X` での `n_nonzero_genes` / `n_nonzero_mt_genes`（per cell）。
- `calculate_pct_mt_by_state`: `qc_preprocessing_state` 別に `pct_mt_for_qc` を計算。
- `add_gene_detection_metrics`: 遺伝子ごとの `n_nonzero_cells` / `pct_nonzero_cells`（per gene）。

すべて `.X` を変更しない。`matrix_sum` は混在スケールで意味がないため計算しない。

In [4]:
def get_mt_mask(adata):
    gene_upper = pd.Index(get_gene_symbols_upper(adata)).astype(str).str.upper()
    return np.asarray(gene_upper.str.startswith("MT-"))


def add_nonzero_metrics(adata):
    """現在の .X の非ゼロ遺伝子数を obs に付ける（.X は変更しない）。"""
    X = adata.X
    if sp.issparse(X):
        n_nz = X.getnnz(axis=1)
    else:
        n_nz = (np.asarray(X) != 0).sum(axis=1)
    adata.obs["n_nonzero_genes"] = np.asarray(n_nz).ravel()

    mt_mask = get_mt_mask(adata)
    if mt_mask.any():
        Xmt = X[:, mt_mask]
        if sp.issparse(Xmt):
            n_mt = Xmt.getnnz(axis=1)
        else:
            n_mt = (np.asarray(Xmt) != 0).sum(axis=1)
        adata.obs["n_nonzero_mt_genes"] = np.asarray(n_mt).ravel()
    else:
        adata.obs["n_nonzero_mt_genes"] = 0
    return adata


def _pct_mt_from_linear_matrix(X, mt_mask):
    """X は linear (count-like) space の前提。行ごとに MT 割合(%)を返す。X は変更しない。"""
    if sp.issparse(X):
        total = np.asarray(X.sum(axis=1)).ravel()
        mt = np.asarray(X[:, mt_mask].sum(axis=1)).ravel() if mt_mask.any() else np.zeros(X.shape[0])
    else:
        Xd = np.asarray(X)
        total = Xd.sum(axis=1)
        mt = Xd[:, mt_mask].sum(axis=1) if mt_mask.any() else np.zeros(Xd.shape[0])
    with np.errstate(divide="ignore", invalid="ignore"):
        pct = np.where(total > 0, 100.0 * mt / total, 0.0)
    return np.asarray(pct).ravel()


def calculate_pct_mt_by_state(adata, state_col="qc_preprocessing_state", obs_key="pct_mt_for_qc"):
    """qc_preprocessing_state 別に pct_mt を計算する。.X は変更しない。

    - raw_count_like / cpm_tpm_like: 現在の .X からそのまま割合を計算（分母は実際の行和）。
    - log_normalized_like: np.expm1 で count-like に戻してから割合を計算。
    - それ以外: NaN（warning）。
    """
    pct = pd.Series(np.nan, index=adata.obs_names, dtype=float)
    mt_mask = get_mt_mask(adata)

    for state in adata.obs[state_col].astype(str).unique():
        cell_mask = (adata.obs[state_col].astype(str) == state).to_numpy()
        sub = adata[cell_mask]
        X = sub.X

        if state in ("raw_count_like", "cpm_tpm_like"):
            vals = _pct_mt_from_linear_matrix(X, mt_mask)
        elif state == "log_normalized_like":
            # y = log1p(c) と仮定し、c = expm1(y) に戻す。疎行列は .data だけに適用。
            if sp.issparse(X):
                Xb = X.copy()
                Xb.data = np.expm1(Xb.data)
            else:
                Xb = np.expm1(np.asarray(X))
            vals = _pct_mt_from_linear_matrix(Xb, mt_mask)
        else:
            print(f"WARNING: unknown preprocessing state '{state}'; pct_mt_for_qc set to NaN")
            vals = np.full(sub.n_obs, np.nan)

        pct.loc[sub.obs_names] = vals

    adata.obs[obs_key] = pct.values
    return adata


def add_gene_detection_metrics(adata):
    """遺伝子ごとの検出細胞数を var に付ける（.X は変更しない）。"""
    X = adata.X
    if sp.issparse(X):
        n_cells_nz = X.getnnz(axis=0)
    else:
        n_cells_nz = (np.asarray(X) != 0).sum(axis=0)
    n_cells_nz = np.asarray(n_cells_nz).ravel()
    adata.var["n_nonzero_cells"] = n_cells_nz
    adata.var["pct_nonzero_cells"] = (100.0 * n_cells_nz / adata.n_obs) if adata.n_obs else 0.0
    return adata


## 4. merge 後の QC 確認指標を計算（outer / inner）

In [5]:
for adata in (merged_outer, merged_inner):
    add_nonzero_metrics(adata)
    calculate_pct_mt_by_state(adata)
    add_gene_detection_metrics(adata)

print("outer obs:", [c for c in ["n_nonzero_genes", "n_nonzero_mt_genes", "pct_mt_for_qc"] if c in merged_outer.obs.columns])
merged_outer.obs[["qc_preprocessing_state", "n_nonzero_genes", "n_nonzero_mt_genes", "pct_mt_for_qc"]].head()


outer obs: ['n_nonzero_genes', 'n_nonzero_mt_genes', 'pct_mt_for_qc']


,qc_preprocessing_state,n_nonzero_genes,n_nonzero_mt_genes,pct_mt_for_qc
cell_uid,,,,
GSE167198__GSE167198_Sample2-matrix-txt-gz_GCACATCCGCCG,raw_count_like,3702,13,1.707646
GSE167198__GSE167198_Sample2-matrix-txt-gz_CCCGAAAAGTAT,raw_count_like,2332,11,1.374237
GSE167198__GSE167198_Sample2-matrix-txt-gz_TCATATTAATAA,raw_count_like,2383,12,1.578776
GSE167198__GSE167198_Sample2-matrix-txt-gz_TCGGACAAAGGC,raw_count_like,2290,10,1.749008
GSE167198__GSE167198_Sample2-matrix-txt-gz_CCATAGGTTACN,raw_count_like,2322,12,2.093137


## MT 遺伝子の検出状況 CSV

`mt_gene_detection_summary.csv`（outer / inner 両方、`merge_type` 列つき）。

In [6]:
def mt_gene_detection_table(adata, merge_type):
    mt_mask = get_mt_mask(adata)
    sub = adata.var.loc[mt_mask]
    return pd.DataFrame({
        "merge_type": merge_type,
        "gene": pd.Index(sub.index).astype(str),
        "n_nonzero_cells": sub["n_nonzero_cells"].to_numpy(),
        "pct_nonzero_cells": sub["pct_nonzero_cells"].to_numpy(),
    })


mt_det = pd.concat(
    [mt_gene_detection_table(merged_outer, "outer"),
     mt_gene_detection_table(merged_inner, "inner")],
    ignore_index=True,
)
mt_det_path = OUT_DIR / "mt_gene_detection_summary.csv"
mt_det.to_csv(mt_det_path, index=False)
print("saved:", mt_det_path, "rows:", len(mt_det))
mt_det.head()


saved: /home/suzuki/Learn/SMA/v2/results/qc_original_scale_pipeline/04d_merged_original_scale/mt_gene_detection_summary.csv rows: 33


,merge_type,gene,n_nonzero_cells,pct_nonzero_cells
0,outer,MT-ATP6,201708,72.720324
1,outer,MT-ATP8,77465,27.927895
2,outer,MT-CO1,229831,82.859306
3,outer,MT-CO2,197425,71.176205
4,outer,MT-CO3,207584,74.838756


## 5. QC 指標を per-cell CSV で保存（追加 QC を決める前）

In [7]:
def per_cell_qc_table(adata):
    cols = [
        "cell_uid", "source_accession", "dataset_id", "Condition", "qc_preprocessing_state",
        "qc_pass_stage1", "qc_reason_stage1",
        "n_nonzero_genes", "n_nonzero_mt_genes", "pct_mt_for_qc",
    ]
    df = pd.DataFrame(index=adata.obs_names)
    for c in cols:
        df[c] = adata.obs[c].values if c in adata.obs.columns else np.nan
    return df.reset_index(drop=True)


outer_metrics = per_cell_qc_table(merged_outer)
inner_metrics = per_cell_qc_table(merged_inner)
outer_metrics.to_csv(OUT_DIR / "merged_outer_qc_metrics_by_cell.csv.gz", index=False, compression="gzip")
inner_metrics.to_csv(OUT_DIR / "merged_inner_qc_metrics_by_cell.csv.gz", index=False, compression="gzip")
print("saved per-cell QC metrics (outer / inner)")
outer_metrics.head()


saved per-cell QC metrics (outer / inner)


,cell_uid,source_accession,dataset_id,Condition,qc_preprocessing_state,qc_pass_stage1,qc_reason_stage1,n_nonzero_genes,n_nonzero_mt_genes,pct_mt_for_qc
0,GSE167198__GSE167198_Sample2-matrix-txt-gz_GCA...,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq,WT,raw_count_like,True,pass,3702,13,1.707646
1,GSE167198__GSE167198_Sample2-matrix-txt-gz_CCC...,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq,WT,raw_count_like,True,pass,2332,11,1.374237
2,GSE167198__GSE167198_Sample2-matrix-txt-gz_TCA...,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq,WT,raw_count_like,True,pass,2383,12,1.578776
3,GSE167198__GSE167198_Sample2-matrix-txt-gz_TCG...,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq,WT,raw_count_like,True,pass,2290,10,1.749008
4,GSE167198__GSE167198_Sample2-matrix-txt-gz_CCA...,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq,WT,raw_count_like,True,pass,2322,12,2.093137


## 6. 可視化

- `merged_cells_per_dataset_condition.png`: dataset x Condition の細胞数（積み上げ棒）
- `merged_condition_fraction_by_dataset.png`: dataset ごとの Condition 割合（積み上げ棒）
- dataset 別 violin: `n_nonzero_genes` / `n_nonzero_mt_genes` / `pct_mt_for_qc`
- preprocessing state 別 violin: 同上
- `merged_mt_gene_n_nonzero_cells.png`: MT 遺伝子ごとの検出細胞数（横棒）

可視化は `merged_outer` を使う（inner は遺伝子が共通部分に絞られ MT 遺伝子が減る可能性があるため）。

In [8]:
def sample_for_plot(adata):
    if adata.n_obs > MAX_PLOT:
        idx = np.sort(RNG.choice(adata.n_obs, size=MAX_PLOT, replace=False))
        return adata[idx].copy()
    return adata


def violin_by(adata_plot, key, groupby, fname):
    if key not in adata_plot.obs.columns or groupby not in adata_plot.obs.columns:
        print(f"  [skip] violin {key} by {groupby} (missing column)")
        return
    try:
        sc.pl.violin(adata_plot, keys=key, groupby=groupby, rotation=90, show=False)
        plt.savefig(PLOT_DIR / fname, dpi=100, bbox_inches="tight")
    except Exception as e:
        print(f"  [warn] violin {key} by {groupby} failed: {e}")
    finally:
        plt.close("all")


obs = merged_outer.obs

# --- dataset x Condition の細胞数（積み上げ棒）と Condition 割合（積み上げ棒） ---
ct = obs.groupby(["dataset_id", "Condition"], observed=True).size().unstack(fill_value=0)

ax = ct.plot(kind="bar", stacked=True, figsize=(max(6, 0.6 * len(ct)), 4))
ax.set_ylabel("n_cells")
ax.set_title("cells per dataset x Condition")
ax.legend(title="Condition", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(PLOT_DIR / "merged_cells_per_dataset_condition.png", dpi=100, bbox_inches="tight")
plt.close("all")

frac = ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
ax = frac.plot(kind="bar", stacked=True, figsize=(max(6, 0.6 * len(frac)), 4))
ax.set_ylabel("fraction in dataset")
ax.set_ylim(0, 1)
ax.set_title("Condition fraction per dataset")
ax.legend(title="Condition", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(PLOT_DIR / "merged_condition_fraction_by_dataset.png", dpi=100, bbox_inches="tight")
plt.close("all")

# --- violin（dataset 別 / preprocessing state 別） ---
plot_adata = sample_for_plot(merged_outer)
violin_by(plot_adata, "n_nonzero_genes", "dataset_id", "merged_n_nonzero_genes_by_dataset.png")
violin_by(plot_adata, "n_nonzero_mt_genes", "dataset_id", "merged_n_nonzero_mt_genes_by_dataset.png")
violin_by(plot_adata, "pct_mt_for_qc", "dataset_id", "merged_pct_mt_for_qc_by_dataset.png")
violin_by(plot_adata, "n_nonzero_genes", "qc_preprocessing_state", "merged_n_nonzero_genes_by_preprocessing_state.png")
violin_by(plot_adata, "n_nonzero_mt_genes", "qc_preprocessing_state", "merged_n_nonzero_mt_genes_by_preprocessing_state.png")
violin_by(plot_adata, "pct_mt_for_qc", "qc_preprocessing_state", "merged_pct_mt_for_qc_by_preprocessing_state.png")

# --- MT 遺伝子ごとの検出細胞数（横棒） ---
mt_var = merged_outer.var.loc[get_mt_mask(merged_outer)]
if len(mt_var):
    order = mt_var["n_nonzero_cells"].sort_values()
    fig, ax = plt.subplots(figsize=(6, max(3, 0.3 * len(order))))
    ax.barh(pd.Index(order.index).astype(str), order.to_numpy())
    ax.set_xlabel("n_nonzero_cells")
    ax.set_title("MT genes: detected cell count (outer)")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "merged_mt_gene_n_nonzero_cells.png", dpi=100, bbox_inches="tight")
    plt.close("all")
else:
    print("no MT genes in merged_outer.var")

print("saved plots ->", PLOT_DIR)


/tmp/ipykernel_1487054/505760539.py:30: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
/tmp/ipykernel_1487054/505760539.py:40: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


saved plots -> /home/suzuki/Learn/SMA/v2/results/qc_original_scale_pipeline/04d_merged_original_scale/plots


## Condition 構成比の確認 CSV

`condition_fraction_by_dataset.csv`: dataset 内の全細胞に対する各 Condition の割合。
QC 後に特定 Condition だけが大きく減っていないか、dataset 間で構成が極端に違わないかを見る。

In [9]:
cf = (
    obs.groupby(["source_accession", "dataset_id", "Condition"], observed=True)
    .size().reset_index(name="n_cells")
)
dataset_total = obs.groupby("dataset_id", observed=True).size()
cf["fraction_in_dataset"] = cf["n_cells"] / cf["dataset_id"].map(dataset_total).to_numpy()
cf = cf[["source_accession", "dataset_id", "Condition", "n_cells", "fraction_in_dataset"]]
cf_path = OUT_DIR / "condition_fraction_by_dataset.csv"
cf.to_csv(cf_path, index=False)
print("saved:", cf_path)
cf.head(20)


saved: /home/suzuki/Learn/SMA/v2/results/qc_original_scale_pipeline/04d_merged_original_scale/condition_fraction_by_dataset.csv


,source_accession,dataset_id,Condition,n_cells,fraction_in_dataset
0,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq,SOD1,2286,0.681371
1,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq,WT,1069,0.318629
2,GSE167327,GSE167327_sc_spinalcord_optn_cd45_enriched_indrop,SOD1,1574,1.000000
3,GSE167331,GSE167331_sc_spinalcord_optn_facs_microglia_sm...,SOD1,326,1.000000
4,GSE173524,GSE173524_sn_lumbar_spinalcord_sod1,SOD1,5590,0.397130
5,GSE173524,GSE173524_sn_lumbar_spinalcord_sod1,WT,8486,0.602870
6,GSE178693,GSE178693_sc_brainstem_trigeminal_sod1_dropseq,SOD1,3398,0.606677
7,GSE178693,GSE178693_sc_brainstem_trigeminal_sod1_dropseq,WT,2203,0.393323
8,GSE206330,GSE206330_sc_cortex_sod1_facs_glia_soupx,SOD1,13303,0.616593
9,GSE206330,GSE206330_sc_cortex_sod1_facs_glia_soupx,WT,8272,0.383407


## 7. 追加 QC 条件を手動で決める

```
ここでプロットとCSVを確認して、必要なら閾値を手動で指定する。
None の条件は適用されない。
```

In [15]:
INTEGRATED_QC = {
    "min_n_nonzero_genes": 200,
    "max_n_nonzero_genes": 7000,
    "max_n_nonzero_mt_genes": 15, #ここだけ条件が追加
    "max_pct_mt_for_qc": 20,
}
INTEGRATED_QC


{'min_n_nonzero_genes': 200,
 'max_n_nonzero_genes': 7000,
 'max_n_nonzero_mt_genes': 15,
 'max_pct_mt_for_qc': 20}

## 8. 追加 QC を実行する

- `qc_pass_integrated`: `INTEGRATED_QC` の条件をすべて満たせば True（全条件 None なら全細胞 True）
- `qc_reason_integrated`: fail reason をセミコロン区切り（pass は "pass"）
- `qc_pass_final = qc_pass_stage1 & qc_pass_integrated`

fail reason: `low_n_nonzero_genes` / `high_n_nonzero_genes` / `high_n_nonzero_mt_genes` / `high_pct_mt_for_qc`

QC は **outer**（全遺伝子があり pct_mt_for_qc が妥当）で判定し、結果を `cell_uid` をキーに inner へコピーする。

In [16]:
def apply_integrated_qc(adata, qc):
    obs = adata.obs
    n = adata.n_obs

    ng = obs["n_nonzero_genes"].to_numpy()
    nmt = obs["n_nonzero_mt_genes"].to_numpy()
    pmt = obs["pct_mt_for_qc"].to_numpy()

    specs = []
    if qc.get("min_n_nonzero_genes") is not None:
        specs.append(("low_n_nonzero_genes", ng < qc["min_n_nonzero_genes"]))
    if qc.get("max_n_nonzero_genes") is not None:
        specs.append(("high_n_nonzero_genes", ng > qc["max_n_nonzero_genes"]))
    if qc.get("max_n_nonzero_mt_genes") is not None:
        specs.append(("high_n_nonzero_mt_genes", nmt > qc["max_n_nonzero_mt_genes"]))
    if qc.get("max_pct_mt_for_qc") is not None:
        specs.append(("high_pct_mt_for_qc", pmt > qc["max_pct_mt_for_qc"]))

    passing = np.ones(n, dtype=bool)
    reasons = pd.Series([""] * n, index=obs.index)
    for label, mask in specs:
        mask = np.asarray(mask, dtype=bool)
        passing = passing & ~mask
        reasons = reasons + np.where(mask, label + ";", "")
    reasons = reasons.str.rstrip(";")
    reasons[reasons == ""] = "pass"

    stage1 = obs["qc_pass_stage1"].astype(bool).to_numpy() if "qc_pass_stage1" in obs.columns else np.ones(n, dtype=bool)

    adata.obs["qc_pass_integrated"] = passing
    adata.obs["qc_reason_integrated"] = reasons.values
    adata.obs["qc_pass_final"] = stage1 & passing
    return adata


# outer で判定
apply_integrated_qc(merged_outer, INTEGRATED_QC)

# inner へ cell_uid をキーにコピー（細胞集合が同じか確認）
outer_cells = set(merged_outer.obs["cell_uid"].astype(str))
inner_cells = set(merged_inner.obs["cell_uid"].astype(str))
if outer_cells != inner_cells:
    print(f"WARNING: outer/inner cell sets differ "
          f"(outer={len(outer_cells)}, inner={len(inner_cells)}, "
          f"only_outer={len(outer_cells - inner_cells)}, only_inner={len(inner_cells - outer_cells)})")
else:
    print("outer/inner cell sets match:", len(outer_cells))

src = merged_outer.obs.set_index("cell_uid")[["qc_pass_integrated", "qc_reason_integrated", "qc_pass_final"]]
inner_key = merged_inner.obs["cell_uid"].astype(str)
merged_inner.obs["qc_pass_integrated"] = inner_key.map(src["qc_pass_integrated"]).to_numpy()
merged_inner.obs["qc_reason_integrated"] = inner_key.map(src["qc_reason_integrated"]).to_numpy()
merged_inner.obs["qc_pass_final"] = inner_key.map(src["qc_pass_final"]).to_numpy()

print("qc_pass_final (outer):", int(merged_outer.obs["qc_pass_final"].sum()), "/", merged_outer.n_obs)
print(merged_outer.obs["qc_reason_integrated"].value_counts().head())


outer/inner cell sets match: 277375
qc_pass_final (outer): 277279 / 277375
qc_reason_integrated
pass                       277279
high_n_nonzero_mt_genes        72
low_n_nonzero_genes            24
Name: count, dtype: int64


## 9. summary CSV を保存

- `merged_dataset_summary.csv`（merge_type / state 別、`matrix_sum` は入れない）
- `merged_condition_summary.csv`（merge_type / Condition 別）
- `merged_final_summary.csv`（integrated QC 前後の細胞数）

dataset / condition summary は **final（qc_pass_final==True）** の保存対象に対して計算する。
final summary は integrated QC 前（全 merge 細胞）と後の比較を入れる。

In [17]:
def _n_genes_by_dataset(adata):
    obs_did = adata.obs["dataset_id"].astype(str).to_numpy()
    out = {}
    for did in np.unique(obs_did):
        sub = adata[obs_did == did]
        Xs = sub.X
        gnnz = Xs.getnnz(axis=0) if sp.issparse(Xs) else (np.asarray(Xs) != 0).sum(axis=0)
        out[did] = int((np.asarray(gnnz).ravel() > 0).sum())
    return out


def dataset_summary(adata, merge_type):
    g = adata.obs.groupby(["source_accession", "dataset_id", "qc_preprocessing_state"], observed=True)
    df = g.agg(
        n_cells=("cell_uid", "size"),
        median_n_nonzero_genes=("n_nonzero_genes", "median"),
        median_n_nonzero_mt_genes=("n_nonzero_mt_genes", "median"),
        median_pct_mt_for_qc=("pct_mt_for_qc", "median"),
    ).reset_index()
    n_genes_map = _n_genes_by_dataset(adata)
    df["n_genes"] = df["dataset_id"].astype(str).map(n_genes_map)
    df.insert(0, "merge_type", merge_type)
    return df[["merge_type", "source_accession", "dataset_id", "qc_preprocessing_state",
               "n_cells", "n_genes", "median_n_nonzero_genes",
               "median_n_nonzero_mt_genes", "median_pct_mt_for_qc"]]


def condition_summary(adata, merge_type):
    g = adata.obs.groupby(["source_accession", "dataset_id", "Condition"], observed=True)
    df = g.agg(
        n_cells=("cell_uid", "size"),
        median_n_nonzero_genes=("n_nonzero_genes", "median"),
        median_n_nonzero_mt_genes=("n_nonzero_mt_genes", "median"),
        median_pct_mt_for_qc=("pct_mt_for_qc", "median"),
    ).reset_index()
    df.insert(0, "merge_type", merge_type)
    return df[["merge_type", "source_accession", "dataset_id", "Condition",
               "n_cells", "median_n_nonzero_genes",
               "median_n_nonzero_mt_genes", "median_pct_mt_for_qc"]]


def final_summary_row(adata_flagged, merge_type):
    obs = adata_flagged.obs
    before = int(obs.shape[0])
    after = int(obs["qc_pass_final"].sum())
    return {
        "merge_type": merge_type,
        "total_cells_before_integrated_qc": before,
        "total_cells_after_integrated_qc": after,
        "fraction_cells_after_integrated_qc": (after / before) if before else float("nan"),
        "n_datasets": int(obs["dataset_id"].nunique()),
        "n_conditions": int(obs["Condition"].nunique()),
        "n_genes": int(adata_flagged.n_vars),
    }


# final（保存対象）= qc_pass_final==True
final_outer = merged_outer[merged_outer.obs["qc_pass_final"].to_numpy()].copy()
final_inner = merged_inner[merged_inner.obs["qc_pass_final"].astype(bool).to_numpy()].copy()

ds_sum = pd.concat([dataset_summary(final_outer, "outer"),
                    dataset_summary(final_inner, "inner")], ignore_index=True)
cond_sum = pd.concat([condition_summary(final_outer, "outer"),
                      condition_summary(final_inner, "inner")], ignore_index=True)
final_sum = pd.DataFrame([final_summary_row(merged_outer, "outer"),
                          final_summary_row(merged_inner, "inner")])

ds_sum.to_csv(OUT_DIR / "merged_dataset_summary.csv", index=False)
cond_sum.to_csv(OUT_DIR / "merged_condition_summary.csv", index=False)
final_sum.to_csv(OUT_DIR / "merged_final_summary.csv", index=False)
print("saved summaries ->", OUT_DIR)
display(final_sum)
display(ds_sum.head())


saved summaries -> /home/suzuki/Learn/SMA/v2/results/qc_original_scale_pipeline/04d_merged_original_scale


,merge_type,total_cells_before_integrated_qc,total_cells_after_integrated_qc,fraction_cells_after_integrated_qc,n_datasets,n_conditions,n_genes
0,outer,277375,277279,0.999654,11,3,34911
1,inner,277375,277279,0.999654,11,3,8863


,merge_type,source_accession,dataset_id,qc_preprocessing_state,n_cells,n_genes,median_n_nonzero_genes,median_n_nonzero_mt_genes,median_pct_mt_for_qc
0,outer,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq,raw_count_like,3323,14888,320.0,8.0,6.804734
1,outer,GSE167327,GSE167327_sc_spinalcord_optn_cd45_enriched_indrop,raw_count_like,1574,15794,799.0,7.5,0.814370
2,outer,GSE167331,GSE167331_sc_spinalcord_optn_facs_microglia_sm...,cpm_tpm_like,326,11542,1674.5,0.0,0.000000
3,outer,GSE173524,GSE173524_sn_lumbar_spinalcord_sod1,raw_count_like,14076,24286,3102.0,0.0,0.000000
4,outer,GSE178693,GSE178693_sc_brainstem_trigeminal_sod1_dropseq,raw_count_like,5542,19502,633.0,7.0,5.638540


## 10. QC 後の outer / inner を保存

標準で使うファイルは `qc_pass_final == True` に絞った以下:

```
data/merged_h5ad/merged_qc_original_scale_outer.h5ad
data/merged_h5ad/merged_qc_original_scale_inner.h5ad
```

`.X` は original-scale のまま（raw_count_like は raw count、cpm_tpm_like は CPM/TPM-like、
log_normalized_like は log normalized）。

QC フラグ付き・未 filter 版も `.flagged.h5ad` として保存する（参照用、必須ではない）。

In [18]:
# 最終（filtered）版
outer_path = MERGED_DIR / "merged_qc_original_scale_outer.h5ad"
inner_path = MERGED_DIR / "merged_qc_original_scale_inner.h5ad"
final_outer.write_h5ad(outer_path)
final_inner.write_h5ad(inner_path)
print("saved:", outer_path, final_outer.shape)
print("saved:", inner_path, final_inner.shape)

# QC フラグ付き・未 filter 版（参照用）
outer_flagged_path = MERGED_DIR / "merged_qc_original_scale_outer.flagged.h5ad"
inner_flagged_path = MERGED_DIR / "merged_qc_original_scale_inner.flagged.h5ad"
merged_outer.write_h5ad(outer_flagged_path)
merged_inner.write_h5ad(inner_flagged_path)
print("saved (flagged):", outer_flagged_path, merged_outer.shape)
print("saved (flagged):", inner_flagged_path, merged_inner.shape)


saved: /home/suzuki/Learn/SMA/v2/data/merged_h5ad/merged_qc_original_scale_outer.h5ad (277279, 34911)
saved: /home/suzuki/Learn/SMA/v2/data/merged_h5ad/merged_qc_original_scale_inner.h5ad (277279, 8863)
saved (flagged): /home/suzuki/Learn/SMA/v2/data/merged_h5ad/merged_qc_original_scale_outer.flagged.h5ad (277375, 34911)
saved (flagged): /home/suzuki/Learn/SMA/v2/data/merged_h5ad/merged_qc_original_scale_inner.flagged.h5ad (277375, 8863)


## （参考）前処理を揃える関数例（実行しない）

merge 後の original-scale AnnData から、探索・可視化用の log-expression AnnData を作るための関数例。
このノートブックでは **実行しない**。実行する場合も入力 `adata` を破壊せず copy を返す設計。

目的:
```
raw_count_like:     normalize_total(target_sum=1e4) -> log1p
cpm_tpm_like:       normalize_total(target_sum=1e4) -> log1p
log_normalized_like: そのまま
```

In [ ]:
# def make_logexpr_copy_from_original_scale(
#     adata,
#     state_col="qc_preprocessing_state",
#     target_sum=1e4,
# ):
#     """
#     Return a new AnnData whose .X is converted to a rough common log-expression space.
#
#     This function does NOT modify the input adata.
#     This output is for exploratory visualization / clustering only.
#     It is NOT a raw-count matrix and must not be used as count input for scVI.
#
#     Rules:
#       - raw_count_like:     normalize_total(target_sum=1e4) then log1p
#       - cpm_tpm_like:       normalize_total(target_sum=1e4) then log1p
#       - log_normalized_like: keep as is
#     """
#     import anndata as ad
#
#     parts = []
#     for state in adata.obs[state_col].astype(str).unique():
#         sub = adata[adata.obs[state_col].astype(str) == state].copy()
#         if state in ["raw_count_like", "cpm_tpm_like"]:
#             sc.pp.normalize_total(sub, target_sum=target_sum)
#             sc.pp.log1p(sub)
#         elif state == "log_normalized_like":
#             pass
#         else:
#             print(f"WARNING: unknown preprocessing state: {state}; keeping .X as is")
#         parts.append(sub)
#
#     out = ad.concat(parts, join="outer", merge="same", label=None, index_unique=None)
#     out.uns["made_from"] = "merged_qc_original_scale"
#     out.uns["logexpr_conversion_note"] = (
#         "raw_count_like and cpm_tpm_like were normalize_total+log1p; "
#         "log_normalized_like was kept as is."
#     )
#     return out
